In [ ]:
import json
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
train_df = pd.read_csv('/kaggle/input/competitions/pkdd-15-taxi-trip-time-prediction-ii/train.csv.zip')
test_df = pd.read_csv('/kaggle/input/competitions/pkdd-15-taxi-trip-time-prediction-ii/test.csv.zip')

In [ ]:
train_df.head()

In [ ]:
def get_trip_metrics(s):
    if not s or s == "[]":
        return 0, 0
    try:
        pts = json.loads(s)
        n = len(pts)
        duration = max(0, (n - 1) * 15)
        return n, duration
    except Exception:
        return 0, 0

In [ ]:
res = [get_trip_metrics(p) for p in train_df["POLYLINE"]]
train_df["n_points"] = [r[0] for r in res]
train_df["duration_s"] = [r[1] for r in res]

In [ ]:
train_df["datetime"] = pd.to_datetime(train_df["TIMESTAMP"], unit="s")
train_df["hour"] = train_df["datetime"].dt.hour
train_df["day_name"] = train_df["datetime"].dt.day_name()

In [ ]:
train_df.groupby("CALL_TYPE").agg(
    total_trips=("TRIP_ID", "count"),
    avg_duration_min=("duration_s", lambda x: (x.mean() / 60)),
    median_duration_min=("duration_s", lambda x: (x.median() / 60)),
    avg_gps_points=("n_points", "mean")
)

In [ ]:
hourly = train_df.groupby("hour").agg(
    trips=("TRIP_ID", "count"),
    avg_duration_min=("duration_s", lambda x: (x.mean() / 60))
)
print(hourly)

In [ ]:
weekly = train_df.groupby("day_name").agg(
    trips=("TRIP_ID", "count"),
    avg_duration_min=("duration_s", lambda x: (x.mean() / 60).round(2))
).loc[["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]]
print(weekly)

In [ ]:
top_stands = train_df[train_df["CALL_TYPE"] == "B"].groupby("ORIGIN_STAND").agg(
    total_pickups=("TRIP_ID", "count"),
    avg_duration_min=("duration_s", lambda x: (x.mean() / 60))
).sort_values("total_pickups", ascending=False).head(10)
print(top_stands)

In [ ]:
driver_stats = train_df.groupby("TAXI_ID").agg(
    trips_completed=("TRIP_ID", "count"),
    avg_trip_duration_min=("duration_s", lambda x: (x.mean() / 60))
).sort_values("trips_completed", ascending=False).head(10)
print(driver_stats)

In [ ]:
pd.crosstab(train_df["day_name"], train_df["CALL_TYPE"], normalize="index").round(3) * 100

In [ ]:
def haversine_dist(lon1, lat1, lon2, lat2):
    r = 6731
    p1, p2 = np.radians(lat1), np.radians(lat2)
    delta_lon = np.radians(lon2 - lon1)
    delta_p = np.radians(lat2 - lat1)

    a = np.sin(delta_p/ 2)**2 + np.cos(p1) * np.cos(p2) * np.sin(delta_lon / 2)**2
    c = 2 * np.arcsin(np.sqrt(a))

    return r * c 

In [ ]:
def calc_dist(poly_str):
    if not poly_str or poly_str == "[]":
        return 0
    try:
        pts = np.array(json.loads(poly_str), dtype=np.float32)
        if len(pts) < 2:
            return 0
        segment_dist = haversine_dist(
            pts[:-1, 0], pts[:-1, 1],
            pts[1:, 0], pts[1:, 1]
        )
        return float(np.sum(segment_dist))
    except Exception:
        return 0

In [ ]:
train_df["trip_distance_km"] = train_df["POLYLINE"].apply(calc_dist)
train_df["datetime"] = pd.to_datetime(train_df["TIMESTAMP"], unit="s")
train_df["hour"] = train_df["datetime"].dt.hour

In [ ]:
# Filter out empty trips and extreme GPS artifacts
clean_trips = train_df[(train_df["trip_distance_km"] > 0.05) & (train_df["trip_distance_km"] <= 60.0)].copy()

In [ ]:
# 4. Analyze distance by CALL_TYPE
print("=== Distance by CALL_TYPE ===")
call_type_stats = clean_trips.groupby("CALL_TYPE")["trip_distance_km"].agg(
    trips="count",
    mean_km="mean",
    median_km="median",
    std_km="std"
).round(2)
print(call_type_stats)

In [ ]:
# 5. Analyze distance by Hour
print("\n=== Distance by Hour of Day ===")
hourly_stats = clean_trips.groupby("hour")["trip_distance_km"].agg(
    trips="count",
    mean_km="mean",
    median_km="median"
).round(2)
print(hourly_stats)

In [ ]:
def build_eta_feature_dataset(train_df, sample_size=150000, seed=42):
    np.random.seed(seed)
    
    valid_mask = (
        (train_df["n_points"] >= 8) &                  
        (train_df["duration_s"] <= 7200) &               
        (train_df["trip_distance_km"] > 0.2) &         
        (train_df["trip_distance_km"] <= 50.0) 
    )
    candidates = train_df[valid_mask]
    
    if len(candidates) > sample_size:
        subset = candidates.sample(n=sample_size, random_state=seed)
    else:
        subset = candidates
    records = []

    for _, row in subset.iterrows():
        try:
            pts = np.array(json.loads(row["POLYLINE"]), dtype=np.float32)
        except Exception:
            continue
            
        n_total = len(pts)
        
        k = np.random.randint(int(n_total * 0.2), max(int(n_total * 0.8), int(n_total * 0.2) + 1))
    
        observed = pts[:k]
        
        dest_lon, dest_lat = pts[-1, 0], pts[-1, 1]
        
        target_remaining_s = float((n_total - k) * 15.0)
        
        pickup_lon, pickup_lat = observed[0, 0], observed[0, 1]
        curr_lon, curr_lat = observed[-1, 0], observed[-1, 1]
        
        dist_to_dest_km = float(haversine_dist(curr_lon, curr_lat, dest_lon, dest_lat))
        
        step_dists = haversine_dist(
            observed[:-1, 0], observed[:-1, 1],
            observed[1:, 0], observed[1:, 1]
        )
        dist_travelled_km = float(np.sum(step_dists))
        displacement_km = float(haversine_dist(pickup_lon, pickup_lat, curr_lon, curr_lat))
        
        tortuosity = (dist_travelled_km / displacement_km) if displacement_km > 0.05 else 1.0
        
        
        elapsed_time_s = float((k - 1) * 15.0)
        avg_speed_kmh = (dist_travelled_km / (elapsed_time_s / 3600.0)) if elapsed_time_s > 0 else 0.0
        recent_speed_kmh = (step_dists[-1] / (15.0 / 3600.0)) if len(step_dists) > 0 else 0.0
        
        hour = int(row["hour"])
        dt = row["datetime"]
        day_of_week = int(dt.dayofweek)
        is_weekend = int(day_of_week >= 5)
        
        is_morning_rush = int(8 <= hour <= 10)    
        is_midday_busy = int(11 <= hour <= 14)     
        is_evening_rush = int(15 <= hour <= 18)  

        sin_hour = float(np.sin(2 * np.pi * hour / 24.0))
        cos_hour = float(np.cos(2 * np.pi * hour / 24.0))
        
        records.append({
            "TRIP_ID": row["TRIP_ID"],
            "CALL_TYPE": row["CALL_TYPE"],
        
            "pickup_lon": pickup_lon,
            "pickup_lat": pickup_lat,
            "curr_lon": curr_lon,
            "curr_lat": curr_lat,
            "dest_lon": dest_lon,
            "dest_lat": dest_lat,
            "dist_to_dest_km": dist_to_dest_km,
            "dist_travelled_km": dist_travelled_km,
            "tortuosity": tortuosity,
          
            "observed_points": k,
            "elapsed_time_s": elapsed_time_s,
            "avg_speed_kmh": avg_speed_kmh,
            "recent_speed_kmh": recent_speed_kmh,
         
            "hour": hour,
            "day_of_week": day_of_week,
            "is_weekend": is_weekend,
            "is_morning_rush": is_morning_rush,
            "is_midday_busy": is_midday_busy,
            "is_evening_rush": is_evening_rush,
            "sin_hour": sin_hour,
            "cos_hour": cos_hour,
         
            "remaining_time_s": target_remaining_s
        })

    return pd.DataFrame(records)


eta_df = build_eta_feature_dataset(train_df, sample_size=150000)

eta_df = eta_df[
    (eta_df["avg_speed_kmh"] >= 1.0) & (eta_df["avg_speed_kmh"] <= 120.0) &
    (eta_df["recent_speed_kmh"] <= 140.0) &
    (eta_df["dist_to_dest_km"] > 0.05) & (eta_df["dist_to_dest_km"] <= 50.0) &
    (eta_df["remaining_time_s"] > 0)
].reset_index(drop=True)

print("Generated Feature Matrix Shape:", eta_df.shape)
eta_df.head()